# 15. The numbers of the paper

From the saved predictions only (`paper.py`): the main table (B = 10,000) and the curves by number of articles (B =
2,000) for every regressor and three baselines without a model of the text; paired comparisons with Holm correction
(main table and curves against TF-IDF; the claims of the text in one family); then the robustness analyses: political
orientation (gap at equal score, the gap the compression alone would produce, calibration by group), who is left out
by the cleaning, and the leakage ladder of step 14.

In [1]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import paper as PA

art, outlets, rows = PA.load()
models = PA.available()
print("models:", models)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


models: ['TF-IDF', 'Qwen3-0.6B', 'Qwen3-8B', 'Mistral-24B', 'ModernBERT-frozen', 'ModernBERT', 'LLM-zero-shot']


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## Main table

In [2]:
T, V_main = PA.main_table(art, outlets, rows, models=models)
T.to_csv(C.OUT / "paper_main_table.csv", index=False)
T.assign(text=[f"{v:.3f} [{a:.3f}; {b:.3f}]" if m in ("Spearman", "Kendall") else f"{v:.1f} [{a:.1f}; {b:.1f}]"
               for v, a, b, m in zip(T.value, T.low, T.high, T.measure)]) \
 .pivot_table(index="model", columns="measure", values="text", aggfunc="first")[PA.MEASURES]

measure,Spearman,Kendall,MAE,within 20
model,,,,
LLM-zero-shot,0.348 [0.222; 0.464],0.246 [0.156; 0.332],24.4 [22.5; 26.2],41.6 [35.4; 48.2]
Mistral-24B,0.735 [0.673; 0.785],0.540 [0.484; 0.593],15.8 [14.3; 17.4],68.1 [61.9; 74.3]
ModernBERT,0.610 [0.523; 0.682],0.435 [0.368; 0.496],17.5 [15.8; 19.3],63.7 [57.5; 69.9]
ModernBERT-frozen,0.619 [0.534; 0.691],0.446 [0.379; 0.507],18.3 [16.6; 20.0],58.8 [52.7; 65.5]
Qwen3-0.6B,0.626 [0.547; 0.692],0.439 [0.377; 0.499],18.8 [17.1; 20.6],57.1 [50.4; 63.7]
Qwen3-8B,0.711 [0.645; 0.765],0.514 [0.457; 0.568],16.3 [14.8; 17.9],69.9 [63.7; 75.7]
TF-IDF,0.674 [0.604; 0.733],0.480 [0.422; 0.537],17.8 [16.2; 19.6],61.5 [54.9; 67.7]
no text (mean),-0.015 [-0.146; 0.114],-0.012 [-0.110; 0.087],22.6 [20.7; 24.5],50.9 [44.2; 57.5]
no text (median),nan [nan; nan],nan [nan; nan],22.1 [19.9; 24.4],54.0 [47.3; 60.6]


In [3]:
T[T.p.notna()][["model", "measure", "diff", "dlo", "dhi", "p", "p_holm", "sig"]].round(4)

,model,measure,diff,dlo,dhi,p,p_holm,sig
4,Qwen3-0.6B,Spearman,-0.0486,-0.1078,0.0095,0.1020,0.8784,False
5,Qwen3-0.6B,Kendall,-0.0409,-0.0909,0.0091,0.1080,0.8784,False
6,Qwen3-0.6B,MAE,0.9602,0.2849,1.6387,0.0052,0.0780,False
7,Qwen3-0.6B,within 20,-4.4248,-8.4071,-0.4425,0.0482,0.6266,False
8,Qwen3-8B,Spearman,0.0368,-0.0029,0.0789,0.0694,0.7634,False
9,Qwen3-8B,Kendall,0.0334,-0.0059,0.0739,0.0976,0.8784,False
10,Qwen3-8B,MAE,-1.5141,-2.2904,-0.7428,0.0002,0.0038,True
11,Qwen3-8B,within 20,8.4071,3.9823,12.8319,0.0002,0.0038,True
12,Mistral-24B,Spearman,0.0605,0.0198,0.1040,0.0038,0.0646,False
13,Mistral-24B,Kendall,0.0602,0.0206,0.1010,0.0038,0.0646,False


## More measures, for the appendix: R², RMSE, AUC of reliable (score ≥ 60) against the others

In [4]:
M = PA.more_measures(art, outlets, rows, models=models)
M.to_csv(C.OUT / "paper_more_measures.csv", index=False)
print(f"reliable outlets (score >= {PA.ME.RELIABLE}): {(outlets.score >= PA.ME.RELIABLE).mean():.1%}")
M.assign(text=[f"{v:.3f} [{a:.3f}; {b:.3f}]" if m != "RMSE" else f"{v:.1f} [{a:.1f}; {b:.1f}]"
               for v, a, b, m in zip(M.value, M.low, M.high, M.measure)]) \
 .pivot_table(index="model", columns="measure", values="text", aggfunc="first")

reliable outlets (score >= 60): 65.0%


measure,AUC (reliable),R2,RMSE
model,,,
LLM-zero-shot,0.665 [0.589; 0.738],-0.085 [-0.261; 0.031],28.2 [26.3; 30.0]
Mistral-24B,0.876 [0.827; 0.920],0.475 [0.406; 0.533],19.6 [17.7; 21.5]
ModernBERT,0.835 [0.781; 0.884],0.341 [0.253; 0.415],22.0 [19.9; 24.0]
ModernBERT-frozen,0.824 [0.769; 0.875],0.308 [0.236; 0.365],22.5 [20.6; 24.5]
Qwen3-0.6B,0.838 [0.782; 0.891],0.268 [0.209; 0.316],23.1 [21.1; 25.1]
Qwen3-8B,0.880 [0.831; 0.923],0.437 [0.368; 0.496],20.3 [18.4; 22.2]
TF-IDF,0.860 [0.807; 0.907],0.339 [0.273; 0.392],22.0 [20.1; 24.0]
no text (mean),0.501 [0.422; 0.577],-0.000 [-0.023; 0.000],27.0 [24.8; 29.3]
no text (median),0.500 [0.500; 0.500],-0.058 [-0.130; -0.013],27.8 [25.1; 30.5]


In [5]:
# Parameters
N_JOBS = 8


## Curves by number of articles

In [6]:
Cv, V_curves = PA.curves(art, outlets, rows, models=models)
Cv.to_csv(C.OUT / "paper_curves.csv", index=False)
Cv.pivot_table(index="model", columns=["measure", "k"], values="value").round(3)[[("Spearman", k) for k in PA.K] + [("MAE", k) for k in PA.K]]

measure           Spearman                                                      MAE                                                        
k                        1      2      3      5     10     20     50    all       1       2       3       5      10      20      50     all
model                                                                                                                                      
LLM-zero-shot        0.209  0.248  0.280  0.305  0.320  0.338  0.343  0.348  25.141  24.725  24.596  24.514  24.444  24.408  24.376  24.377
Mistral-24B          0.616  0.670  0.685  0.712  0.728  0.729  0.732  0.735  17.323  16.486  16.418  16.098  15.869  15.839  15.812  15.793
ModernBERT           0.450  0.503  0.545  0.572  0.591  0.604  0.608  0.610  19.532  18.676  18.217  17.948  17.701  17.613  17.569  17.542
ModernBERT-frozen    0.384  0.473  0.498  0.557  0.589  0.609  0.613  0.619  20.595  19.431  19.129  18.710  18.483  18.373  18.332  18.255
Qwen3-0.6B           0.333  0.436  0.470  0.529  0.577  0.605  0.620  0.626  21.294  19.974  19.640  19.254  18.968  18.876  18.835  18.785
Qwen3-8B             0.505  0.589  0.620  0.660  0.692  0.700  0.708  0.711  19.441  17.672  17.462  16.838  16.456  16.438  16.334  16.311
TF-IDF               0.393  0.505  0.555  0.592  0.637  0.656  0.670  0.674  20.476  19.031  18.548  18.307  18.000  17.942  17.879  17.825

In [7]:
Cv[Cv.p.notna()].pivot_table(index="model", columns=["measure", "k"], values="beats_tfidf", aggfunc="first")

measure              MAE                                                  Spearman                                                 
k                      1     10      2     20      3      5     50    all        1     10      2     20      3      5     50    all
model                                                                                                                              
LLM-zero-shot      False  False  False  False  False  False  False  False    False  False  False  False  False  False  False  False
Mistral-24B         True   True   True   True   True   True   True   True     True   True   True   True   True   True   True   True
ModernBERT         False  False  False  False  False  False  False  False    False  False  False  False  False  False  False  False
ModernBERT-frozen  False  False  False  False  False  False  False  False    False  False  False  False  False  False  False  False
Qwen3-0.6B         False  False  False  False  False  False  False  False    False  False  False  False  False  False  False  False
Qwen3-8B            True   True   True   True   True   True   True   True     True   True   True  False   True   True  False  False

## The claims of the text (one Holm family)

In [8]:
Cl = PA.claims(V_main, V_curves, models)
Cl.to_csv(C.OUT / "paper_claims.csv", index=False)
Cl.round(4)

,claim,measure,first,second,diff,dlo,dhi,p,p_holm,first better
0,Mistral with 5 articles vs TF-IDF with all,Spearman,0.7119,0.6744,0.0375,-0.0022,0.0815,0.065,0.325,False
1,Mistral with 5 articles vs TF-IDF with all,MAE,16.0985,17.8247,-1.7262,-2.5849,-0.8696,0.001,0.006,True
2,Mistral vs Qwen3-8B,Spearman,0.7348,0.7112,0.0237,-0.0081,0.0563,0.134,0.444,False
3,Mistral vs Qwen3-8B,MAE,15.7934,16.3106,-0.5172,-1.1587,0.1089,0.111,0.444,False
4,Qwen3-8B vs Qwen3-0.6B (size),Spearman,0.7112,0.6258,0.0854,0.0393,0.1355,0.000,0.000,True
5,Qwen3-8B vs Qwen3-0.6B (size),MAE,16.3106,18.7849,-2.4743,-3.2394,-1.6520,0.000,0.000,True
6,ModernBERT frozen vs trained,Spearman,0.6194,0.6104,0.0090,-0.0459,0.0654,0.701,0.701,False
7,ModernBERT frozen vs trained,MAE,18.2552,17.5421,0.7131,-0.2584,1.6110,0.157,0.444,False
8,Mistral frozen + ridge vs zero-shot,Spearman,0.7348,0.3479,0.3869,0.2707,0.5048,0.000,0.000,True
9,Mistral frozen + ridge vs zero-shot,MAE,15.7934,24.3770,-8.5836,-10.2521,-6.9414,0.000,0.000,True


## Where the models err: levels and political orientation

In [9]:
E = PA.errors(art, outlets, rows, models=models)
E.to_csv(C.OUT / "paper_errors.csv", index=False)
E[E.quantity.isin(["signed error", "signed error at equal score", "Spearman inside the group"])] \
 .pivot_table(index="model", columns=["quantity", "group"], values="value").round(2)

quantity          Spearman inside the group             signed error                                                signed error at equal score            
group                                  Left Right  none         Left Right     l1     l2    l3     l4     l5   none                        Left Right  none
model                                                                                                                                                      
LLM-zero-shot                          0.30  0.29  0.23       -22.86  1.18  32.01   5.81 -8.52 -27.72 -38.45 -19.26                       -1.02 -1.12  3.04
Mistral-24B                            0.38  0.62  0.63        -2.85  4.12  30.97   8.94 -2.67 -10.88 -18.88  -7.14                        5.23 -4.34  1.25
ModernBERT                             0.23  0.52  0.52        -4.70  7.04  35.04  10.61  0.58 -10.44 -22.09  -6.16                        3.30 -3.57  2.19
ModernBERT-frozen                      0.30  0.50  0.48        -7.91  8.09  37.39  11.32 -0.75 -14.12 -24.62  -9.58                        2.33 -2.22  1.04
Qwen3-0.6B                             0.29  0.48  0.45        -8.26  8.56  39.35  11.41 -2.02 -13.63 -26.43  -9.93                        2.35 -2.26  1.09
Qwen3-8B                               0.34  0.59  0.56        -3.91  5.04  32.87   8.24 -3.31 -10.03 -20.87  -7.36                        4.48 -3.89  1.36
TF-IDF                                 0.27  0.56  0.47        -6.70  6.69  36.49  11.05 -2.24 -12.81 -25.39  -8.89                        3.39 -3.27  1.58

In [10]:
E[E.group == "all"].pivot_table(index="model", columns="quantity", values="value")

quantity,"Kruskal-Wallis p, at equal score","Kruskal-Wallis p, raw",slope of the signed error on the score
model,,,
LLM-zero-shot,3.043035e-06,2.348032e-11,-0.914031
Mistral-24B,8.117827e-10,3.598494e-04,-0.626075
ModernBERT,6.591587e-05,1.117346e-05,-0.704500
ModernBERT-frozen,5.267905e-04,1.486644e-08,-0.777685
Qwen3-0.6B,6.561270e-05,1.335420e-08,-0.811256
Qwen3-8B,6.164391e-08,5.791894e-05,-0.655643
TF-IDF,1.284942e-07,5.231022e-07,-0.759256


## Per article and per outlet

In [11]:
AO = PA.article_vs_outlet(art, outlets, rows, models=models)
AO.to_csv(C.OUT / "paper_article_vs_outlet.csv", index=False)
AO.set_index(["model", "unit"]).round(3)

n  Spearman  Kendall     MAE  within 20  lowest prediction  highest prediction
model             unit                                                                                       
TF-IDF            article  25524     0.429    0.296  20.445     56.578            -21.953             136.660
                  outlet     226     0.674    0.480  17.825     61.504             37.280              98.280
Qwen3-0.6B        article  25524     0.368    0.252  21.320     54.137              9.173             130.606
                  outlet     226     0.626    0.439  18.785     57.080             48.691              90.050
Qwen3-8B          article  25524     0.525    0.366  19.444     59.250             -9.814             156.296
                  outlet     226     0.711    0.514  16.311     69.912             38.331              99.016
Mistral-24B       article  25524     0.614    0.434  17.849     63.525             -2.977             122.321
                  outlet     226     0.735    0.540  15.793     68.142             37.268              95.996
ModernBERT-frozen article  25524     0.403    0.279  20.867     54.995             -6.222             128.290
                  outlet     226     0.619    0.446  18.255     58.850             43.533              91.069
ModernBERT        article  25524     0.458    0.318  20.352     58.004            -10.980             110.449
                  outlet     226     0.610    0.435  17.542     63.717              3.068              94.046
LLM-zero-shot     article  25524     0.233    0.185  24.255     43.849              0.000              95.000
                  outlet     226     0.348    0.246  24.377     41.593             33.598              74.895

## Political orientation

In [12]:
O = PA.orientation(art, outlets, rows, models=models)
O.to_csv(C.OUT / "paper_orientation.csv", index=False)
O.assign(text=[f"{v:+.1f} [{a:+.1f}; {b:+.1f}]" for v, a, b in zip(O.value, O.low, O.high)]) \
 .pivot_table(index="model", columns="quantity", values="text", aggfunc="first")

quantity,calibration gap,gap from compression,gap levels,gap linear,gap spline
model,,,,,
LLM-zero-shot,+24.8 [+17.8; +31.4],+18.4 [+14.4; +21.3],-0.2 [-2.4; +2.1],+0.1 [-1.9; +2.0],-0.0 [-1.9; +1.8]
Mistral-24B,+0.8 [-7.5; +8.9],+12.6 [+9.9; +14.6],+8.7 [+5.5; +11.7],+9.6 [+6.7; +12.3],+8.8 [+5.8; +11.3]
ModernBERT,+12.8 [+4.8; +20.6],+14.2 [+11.1; +16.5],+5.9 [+2.5; +9.3],+6.9 [+3.8; +10.0],+6.2 [+3.2; +8.9]
ModernBERT-frozen,+12.7 [+4.8; +20.2],+15.6 [+12.4; +18.0],+3.9 [+1.1; +6.6],+4.5 [+2.3; +6.8],+4.2 [+1.9; +6.3]
Qwen3-0.6B,+13.4 [+5.3; +21.0],+16.3 [+12.9; +18.8],+3.6 [+1.2; +6.0],+4.6 [+2.6; +6.6],+3.9 [+2.0; +5.7]
Qwen3-8B,+4.8 [-3.4; +12.8],+13.2 [+10.4; +15.4],+7.0 [+3.8; +10.1],+8.4 [+5.5; +11.3],+7.1 [+4.1; +9.8]
TF-IDF,+7.1 [-0.9; +15.1],+15.3 [+12.1; +17.7],+5.8 [+3.0; +8.4],+6.7 [+4.3; +8.9],+6.0 [+3.6; +8.1]
orientation only,NaN,+15.4 [+12.2; +17.9],+19.3 [+15.5; +22.5],+20.5 [+17.5; +23.3],+19.3 [+15.5; +21.8]


## Who is left out by the cleaning

In [13]:
f = C.OUT / "ladder" / "outlets_by_step.csv"
if f.exists():
    S, tests = PA.selection(pd.read_csv(f, index_col="domain"))
    S.to_csv(C.OUT / "paper_selection.csv", index=False)
    display(S.round(1)); print(tests)
else:
    print("run step 14 first")

,last step,outlets,mean score,% l1,% l2,% l3,% l4,% l5,% Left,% Right,% none,% US
0,sent to the scraper,619,41.4,43.3,35.1,5.0,13.2,3.4,7.4,49.9,42.6,68.2
1,at least one article,6,78.2,0.0,33.3,0.0,50.0,16.7,0.0,66.7,33.3,50.0
2,text extracted,91,73.6,11.0,14.3,8.8,45.1,20.9,19.8,22.0,58.2,15.4
3,English,12,55.0,25.0,16.7,25.0,25.0,8.3,8.3,66.7,25.0,66.7
4,"100-1,500 words",34,56.4,20.6,41.2,5.9,23.5,8.8,29.4,26.5,44.1,76.5
5,from 2025,170,37.0,88.8,1.8,1.8,6.5,1.2,7.1,5.3,87.6,98.2
6,no exact duplicates,14,27.9,92.9,0.0,7.1,0.0,0.0,0.0,28.6,71.4,100.0
7,no near duplicates,60,57.5,21.7,35.0,10.0,25.0,8.3,26.7,35.0,38.3,76.7
8,>= 10 articles,61,47.8,13.1,68.9,6.6,9.8,1.6,36.1,59.0,4.9,96.7
9,one outlet per group,226,68.5,15.5,19.0,13.3,35.0,17.3,30.1,43.8,26.1,79.2


{'score (Mann-Whitney p)': np.float64(5.409168114148103e-26), 'orientation (chi-square p)': np.float64(8.915388070190938e-15)}


## The leakage ladder (step 14)

In [14]:
f = C.OUT / "paper_ladder.csv"
if f.exists():
    L = pd.read_csv(f)
    display(L.pivot_table(index=["outlets", "rung"], columns=["measure", "model"], values="value").round(3))
    for name in ["paper_sisters_similarity", "paper_sisters_gain", "paper_owner_scores"]:
        display(pd.read_csv(C.OUT / f"{name}.csv").round(3))
else:
    print("run step 14 first")

measure                                MAE                     Spearman                
model                          Mistral-24B Qwen3-8B  TF-IDF Mistral-24B Qwen3-8B TF-IDF
outlets          rung                                                                  
all              L2                 15.734   15.984  17.962       0.734    0.724  0.673
                 L3                 15.702   15.940  17.698       0.734    0.724  0.679
                 L4                 15.793   16.311  17.825       0.735    0.711  0.674
                 article split       8.652   10.690  10.390       0.924    0.884  0.904
                 sisters            14.926   15.651  17.126       0.732    0.733  0.712
with a sister    L2                 18.344   18.419  19.923       0.732    0.681  0.664
                 sisters             9.658   11.557   8.301       0.943    0.908  0.918
without a sister L2                 15.319   15.597  17.651       0.739    0.734  0.667
                 sisters            15.763   16.301  18.529       0.698    0.707  0.676

,outlets,n,similarity without the sisters,with the sisters,% closest training article is a sister's,% near copy in a sister
0,with a sister,31,0.185,0.603,69.634,49.808
1,without a sister,195,0.163,0.176,25.086,0.559


,model,outlets,n,Spearman,low,high
0,TF-IDF,all,226,0.267,0.133,0.391
1,TF-IDF,with a sister,31,0.599,0.301,0.804
2,Qwen3-8B,all,226,0.116,-0.023,0.251
3,Qwen3-8B,with a sister,31,0.247,-0.108,0.570
4,Mistral-24B,all,226,0.194,0.055,0.323
5,Mistral-24B,with a sister,31,0.319,-0.028,0.623


,pairs,outlets,mean |difference|,median |difference|,% the same score
0,the same owner (paper outlet and its sisters),31,3.833,0.0,74.194
1,any two of these outlets,120,29.551,25.0,6.681


## Other splits of the outlets (step 16)

In [15]:
f = C.OUT / "paper_splits.csv"
if f.exists():
    S = pd.read_csv(f)
    display(S.groupby("model")[["Spearman", "MAE", "Spearman - TF-IDF", "MAE - TF-IDF"]].agg(["mean", "std", "min", "max"]).round(3))
else:
    print("run step 16 first")

Spearman                          MAE                        Spearman - TF-IDF                      MAE - TF-IDF                     
                mean    std    min    max    mean    std     min     max              mean    std    min    max         mean    std    min    max
model                                                                                                                                            
Mistral-24B    0.727  0.011  0.716  0.742  15.850  0.166  15.662  16.071             0.053  0.015  0.030  0.069       -2.176  0.110 -2.285 -2.031
Qwen3-8B       0.700  0.011  0.688  0.713  16.343  0.151  16.146  16.517             0.026  0.013  0.007  0.040       -1.682  0.146 -1.896 -1.514
TF-IDF         0.674  0.010  0.660  0.690  18.026  0.171  17.825  18.220             0.000  0.000  0.000  0.000        0.000  0.000  0.000  0.000